# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [31]:
# Write your code below.
import pandas as pd
import os
import sys
from glob import glob
from dotenv import load_dotenv

%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [41]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
PRICE_DATA = os.getenv("PRICE_DATA")


print("PRICE_DATA =", PRICE_DATA)
print("Directory exists:", os.path.isdir(PRICE_DATA))


PRICE_DATA = ../../05_src/data/prices/
Directory exists: True


In [62]:
PRICE_DATA

'../../05_src/data/prices/'

In [65]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
if not PRICE_DATA:  
    raise ValueError("PRICE_DATA environment variable is not set.")

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive=True)
print("Parquet files found:", parquet_files)

Parquet files found: ['../../05_src/data/prices/ATRI/ATRI_1982/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_1982/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_1985/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_1985/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_1984/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_1984/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_1983/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_1983/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_2019/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_2019/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_2010/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_2010/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_2017/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_2017/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_2016/part.0.parquet', '../../05_src/data/prices/ATRI/ATRI_2016/part.1.parquet', '../../05_src/data/prices/ATRI/ATRI_2011/part.0.pa

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [80]:
ddf = dd.read_parquet(parquet_files)
ddf['Date'] = dd.to_datetime(ddf['Date'])

#ddf = ddf.set_index(('Date'))

#def add_lags(df):
#ddf = ddf.sort_index()
ddf['Close_lag_1'] = ddf['Close'].shift(1)
ddf['Adj_Close_lag_1'] = ddf['Adj Close'].shift(1)
ddf['Returns'] = ddf['Close']/ddf['Close_lag_1']-1
ddf['High_Low_Range'] = ddf['High']-ddf['Low']
#    return df

#ddf_with_lags = ddf.groupby('ticker').apply(add_lags, meta={
#    'Open': 'f8',
#    'High': 'f8',
#    'Low': 'f8',
#    'Close': 'f8',
#    'Adj_Close': 'f8',
#    'Volume': 'i8',
#    'ticker': 'object',
#    'Close_lag_1': 'f8',
#    'Adj_Close_lag_1': 'f8'
#})

print(ddf.head())

            Date       Open       High        Low      Close  Adj Close  \
38739 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
38740 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
38741 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
38742 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
38743 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

           Volume source ticker  Year  Close_lag_1  Adj_Close_lag_1   Returns  \
38739  62546300.0  A.csv      A  1999          NaN              NaN       NaN   
38740  15234100.0  A.csv      A  1999    31.473534        27.068665 -0.082386   
38741   6577800.0  A.csv      A  1999    28.880543        24.838577  0.089783   
38742   5975600.0  A.csv      A  1999    31.473534        27.068665 -0.090909   
38743   4843200.0  A.csv      A  1999    28.612303        24.607880  0.026563   

       High_Low_Range  
38739        7.153078  
38740        2

In [81]:
ddf.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'Returns',
       'High_Low_Range'],
      dtype='object')

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [82]:
# Write your code below.
# Convert Dask dataframe to pandas dataframe
pdf = ddf.compute()


In [86]:
pdf.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year', 'Close_lag_1', 'Adj_Close_lag_1', 'Returns',
       'High_Low_Range'],
      dtype='object')

In [87]:
# Ensure 'Date' is datetime and sorted (if not already)
#pdf['Date'] = pd.to_datetime(pdf['Date'])
#pdf = pdf.sort_values(['ticker', 'Date'])

# Calculate 10-day moving average of 'returns' grouped by ticker
pdf['Returns_ma_10'] = pdf['Returns'].rolling(window=10).mean()

# Preview result
#print(pdf[['ticker', 'Date', 'returns', 'returns_ma_10']].head(15))

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it’s not strictly necessary.

Generally, yes, it’s better to do it in Dask if your data is large.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.